# Machine Learning Pipeline for Loan Prediction
## Namal University Mianwali - Introduction to Data Science
### Assignment 1
**Instructor:** Dr. Israr Ali Khan  
**Date:** April 10, 2026

## Import Required Libraries

In [3]:
# Import necessary libraries for data manipulation and machine learning
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

---
## Task 1: Load and Preprocess Dataset

### 1.1 Load Dataset

In [4]:
# Load the loan prediction dataset using pandas
df = pd.read_csv('loan-prediction-dataset.csv')

# Display first few rows of the dataset
print("First 5 rows of the dataset:")
print(df.head())

# Display dataset shape and information
print("\nDataset Shape:", df.shape)
print("\nDataset Columns:")
print(df.columns.tolist())

First 5 rows of the dataset:
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural      

### 1.2 Check for Missing Values

In [5]:
# Check for missing values in each column
print("Missing values in each column:")
print(df.isnull().sum())

# Calculate percentage of missing values
print("\nPercentage of missing values:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage)

Missing values in each column:
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

Percentage of missing values:
Loan_ID              0.000000
Gender               2.117264
Married              0.488599
Dependents           2.442997
Education            0.000000
Self_Employed        5.211726
ApplicantIncome      0.000000
CoapplicantIncome    0.000000
LoanAmount           3.583062
Loan_Amount_Term     2.280130
Credit_History       8.143322
Property_Area        0.000000
Loan_Status          0.000000
dtype: float64


### 1.3 Handle Missing Values

In [6]:
# Handle missing values for numerical columns - fill with median
numerical_cols = ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']
for col in numerical_cols:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# Handle missing values for categorical columns - fill with mode
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed']
for col in categorical_cols:
    if col in df.columns:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Verify missing values are handled
print("Missing values after handling:")
print(df.isnull().sum())

Missing values after handling:
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


### 1.4 Remove Duplicate Rows

In [7]:
# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

print("Dataset shape after removing duplicates:", df.shape)

Number of duplicate rows: 0
Dataset shape after removing duplicates: (614, 13)


### 1.5 Display Clean Dataset

In [8]:
# Display clean dataset
print("Clean Dataset:")
print(df.head(10))
print("\nDataset Statistics:")
print(df.describe())

Clean Dataset:
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   
5  LP001011   Male     Yes          2      Graduate           Yes   
6  LP001013   Male     Yes          0  Not Graduate            No   
7  LP001014   Male     Yes         3+      Graduate            No   
8  LP001018   Male     Yes          2      Graduate            No   
9  LP001020   Male     Yes          1      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0       128.0             360.0   
1             4583             1508.0       128.0             360.0   
2           

### 1.6 Encode Categorical Variables

In [9]:
# Create a copy of the dataframe for preprocessing
df_processed = df.copy()

# Drop Loan_ID as it's not useful for prediction
df_processed = df_processed.drop('Loan_ID', axis=1)

# Initialize label encoder
label_encoder = LabelEncoder()

# Encode categorical variables
categorical_columns = ['Gender', 'Married', 'Dependents', 'Education', 
                       'Self_Employed', 'Property_Area', 'Loan_Status']

for col in categorical_columns:
    if col in df_processed.columns:
        df_processed[col] = label_encoder.fit_transform(df_processed[col].astype(str))

# Display preprocessed dataset
print("Preprocessed Dataset (Encoded):")
print(df_processed.head(10))

Preprocessed Dataset (Encoded):
   Gender  Married  Dependents  Education  Self_Employed  ApplicantIncome  \
0       1        0           0          0              0             5849   
1       1        1           1          0              0             4583   
2       1        1           0          0              1             3000   
3       1        1           0          1              0             2583   
4       1        0           0          0              0             6000   
5       1        1           2          0              1             5417   
6       1        1           0          1              0             2333   
7       1        1           3          0              0             3036   
8       1        1           2          0              0             4006   
9       1        1           1          0              0            12841   

   CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
0                0.0       128.0             360.0

### 1.7 Separate Features and Target Variable

In [10]:
# Separate features (X) and target variable (y)
# Loan_Status is the label column
X = df_processed.drop('Loan_Status', axis=1)
y = df_processed['Loan_Status']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

Features shape: (614, 11)
Target shape: (614,)

Feature columns:
['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']


### 1.8 Feature Scaling

In [11]:
# The scaler is created here, but fitted after the train/test split
# (next section), so it only learns from the training data.
scaler = StandardScaler()

### 1.9 Split Dataset into Train and Test Sets

In [12]:
# Split the dataset into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

# Fit the scaler on the training data only, then apply it to both sets
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("\nTarget distribution in training set:")
print(y_train.value_counts())
print("\nTarget distribution in testing set:")
print(y_test.value_counts())

Training set size: (491, 11)
Testing set size: (123, 11)

Target distribution in training set:
Loan_Status
1    342
0    149
Name: count, dtype: int64

Target distribution in testing set:
Loan_Status
1    80
0    43
Name: count, dtype: int64


---
## Task 2: Apply KNN Classifier

### 2.1 KNN with K=3

In [13]:
# Initialize KNN classifier with K=3
knn_3 = KNeighborsClassifier(n_neighbors=3)

# Train the model
knn_3.fit(X_train, y_train)

# Make predictions
y_pred_knn_3 = knn_3.predict(X_test)

print("KNN (K=3) - Predictions:")
print("First 20 predictions:", y_pred_knn_3[:20])
print("First 20 actual values:", y_test.values[:20])

KNN (K=3) - Predictions:
First 20 predictions: [1 1 1 0 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1]
First 20 actual values: [1 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1]


### 2.2 KNN with K=5

In [14]:
# Initialize KNN classifier with K=5
knn_5 = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_5.fit(X_train, y_train)

# Make predictions
y_pred_knn_5 = knn_5.predict(X_test)

print("KNN (K=5) - Predictions:")
print("First 20 predictions:", y_pred_knn_5[:20])
print("First 20 actual values:", y_test.values[:20])

KNN (K=5) - Predictions:
First 20 predictions: [1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1]
First 20 actual values: [1 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1]


### 2.3 KNN with K=7

In [15]:
# Initialize KNN classifier with K=7
knn_7 = KNeighborsClassifier(n_neighbors=7)

# Train the model
knn_7.fit(X_train, y_train)

# Make predictions
y_pred_knn_7 = knn_7.predict(X_test)

print("KNN (K=7) - Predictions:")
print("First 20 predictions:", y_pred_knn_7[:20])
print("First 20 actual values:", y_test.values[:20])

KNN (K=7) - Predictions:
First 20 predictions: [1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1]
First 20 actual values: [1 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1]


---
## Task 3: Apply Naive Bayes Classifier

### 3.1 Gaussian Naive Bayes

In [16]:
# Initialize Gaussian Naive Bayes classifier
gnb = GaussianNB()

# Train the model
gnb.fit(X_train, y_train)

# Make predictions
y_pred_gnb = gnb.predict(X_test)

print("Gaussian Naive Bayes - Predictions:")
print("First 20 predictions:", y_pred_gnb[:20])
print("First 20 actual values:", y_test.values[:20])

Gaussian Naive Bayes - Predictions:
First 20 predictions: [1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1]
First 20 actual values: [1 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1]


---
## Task 4: Model Evaluation

### 4.1 Evaluation Metrics for KNN (K=3)

In [17]:
# Calculate evaluation metrics for KNN with K=3
accuracy_knn_3 = accuracy_score(y_test, y_pred_knn_3)
precision_knn_3 = precision_score(y_test, y_pred_knn_3, average='weighted')
recall_knn_3 = recall_score(y_test, y_pred_knn_3, average='weighted')
f1_knn_3 = f1_score(y_test, y_pred_knn_3, average='weighted')

print("="*50)
print("KNN (K=3) - Evaluation Metrics")
print("="*50)
print(f"Accuracy:  {accuracy_knn_3:.4f}")
print(f"Precision: {precision_knn_3:.4f}")
print(f"Recall:    {recall_knn_3:.4f}")
print(f"F1 Score:  {f1_knn_3:.4f}")
print("="*50)

KNN (K=3) - Evaluation Metrics
Accuracy:  0.7480
Precision: 0.7484
Recall:    0.7480
F1 Score:  0.7256


### 4.2 Evaluation Metrics for KNN (K=5)

In [18]:
# Calculate evaluation metrics for KNN with K=5
accuracy_knn_5 = accuracy_score(y_test, y_pred_knn_5)
precision_knn_5 = precision_score(y_test, y_pred_knn_5, average='weighted')
recall_knn_5 = recall_score(y_test, y_pred_knn_5, average='weighted')
f1_knn_5 = f1_score(y_test, y_pred_knn_5, average='weighted')

print("="*50)
print("KNN (K=5) - Evaluation Metrics")
print("="*50)
print(f"Accuracy:  {accuracy_knn_5:.4f}")
print(f"Precision: {precision_knn_5:.4f}")
print(f"Recall:    {recall_knn_5:.4f}")
print(f"F1 Score:  {f1_knn_5:.4f}")
print("="*50)

KNN (K=5) - Evaluation Metrics
Accuracy:  0.7561
Precision: 0.7676
Recall:    0.7561
F1 Score:  0.7289


### 4.3 Evaluation Metrics for KNN (K=7)

In [19]:
# Calculate evaluation metrics for KNN with K=7
accuracy_knn_7 = accuracy_score(y_test, y_pred_knn_7)
precision_knn_7 = precision_score(y_test, y_pred_knn_7, average='weighted')
recall_knn_7 = recall_score(y_test, y_pred_knn_7, average='weighted')
f1_knn_7 = f1_score(y_test, y_pred_knn_7, average='weighted')

print("="*50)
print("KNN (K=7) - Evaluation Metrics")
print("="*50)
print(f"Accuracy:  {accuracy_knn_7:.4f}")
print(f"Precision: {precision_knn_7:.4f}")
print(f"Recall:    {recall_knn_7:.4f}")
print(f"F1 Score:  {f1_knn_7:.4f}")
print("="*50)

KNN (K=7) - Evaluation Metrics
Accuracy:  0.7805
Precision: 0.8072
Recall:    0.7805
F1 Score:  0.7542


### 4.4 Evaluation Metrics for Gaussian Naive Bayes

In [20]:
# Calculate evaluation metrics for Gaussian Naive Bayes
accuracy_gnb = accuracy_score(y_test, y_pred_gnb)
precision_gnb = precision_score(y_test, y_pred_gnb, average='weighted')
recall_gnb = recall_score(y_test, y_pred_gnb, average='weighted')
f1_gnb = f1_score(y_test, y_pred_gnb, average='weighted')

print("="*50)
print("Gaussian Naive Bayes - Evaluation Metrics")
print("="*50)
print(f"Accuracy:  {accuracy_gnb:.4f}")
print(f"Precision: {precision_gnb:.4f}")
print(f"Recall:    {recall_gnb:.4f}")
print(f"F1 Score:  {f1_gnb:.4f}")
print("="*50)

Gaussian Naive Bayes - Evaluation Metrics
Accuracy:  0.7805
Precision: 0.8072
Recall:    0.7805
F1 Score:  0.7542


### 4.5 Comparison of All Models

In [22]:
# Create a comparison dataframe for all models
comparison_df = pd.DataFrame({
    'Model': ['KNN (K=3)', 'KNN (K=5)', 'KNN (K=7)', 'Gaussian Naive Bayes'],
    'Accuracy': [accuracy_knn_3, accuracy_knn_5, accuracy_knn_7, accuracy_gnb],
    'Precision': [precision_knn_3, precision_knn_5, precision_knn_7, precision_gnb],
    'Recall': [recall_knn_3, recall_knn_5, recall_knn_7, recall_gnb],
    'F1 Score': [f1_knn_3, f1_knn_5, f1_knn_7, f1_gnb]
})

print("="*80)
print("COMPARISON OF ALL MODELS")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Find the best model based on accuracy
best_accuracy = comparison_df['Accuracy'].max()
best_models = comparison_df[comparison_df['Accuracy'] == best_accuracy]['Model'].tolist()
print(f"\nBest Model(s): {', '.join(best_models)} with Accuracy: {best_accuracy:.4f}")

COMPARISON OF ALL MODELS
               Model  Accuracy  Precision   Recall  F1 Score
           KNN (K=3)  0.747967   0.748358 0.747967  0.725608
           KNN (K=5)  0.756098   0.767621 0.756098  0.728918
           KNN (K=7)  0.780488   0.807175 0.780488  0.754213
Gaussian Naive Bayes  0.780488   0.807175 0.780488  0.754213

Best Model(s): KNN (K=7), Gaussian Naive Bayes with Accuracy: 0.7805


---
## Summary

This notebook demonstrates a complete machine learning pipeline for loan prediction:

1. **Data Preprocessing**: Loaded dataset, handled missing values, removed duplicates, encoded categorical variables, and scaled features
2. **KNN Classification**: Applied K-Nearest Neighbors with K=3, 5, and 7
3. **Naive Bayes Classification**: Applied Gaussian Naive Bayes classifier
4. **Model Evaluation**: Calculated accuracy, precision, recall, and F1 score for all models

The comparison table above shows the performance of all models, helping identify the best classifier for loan prediction.